In [ ]:
import pandas as pd
import altair as alt
import yaml
from testing import TESTING_DIR, logger
from testing.r_and_d.boolean_queries.query_tester import reference_df, get_question_id

BOOL_DIR = TESTING_DIR / "r_and_d/boolean_queries/outputs/"
CONFIG_PATH = TESTING_DIR / "r_and_d/boolean_queries/config.yaml"            
OUTPUTS_DIR = TESTING_DIR / "r_and_d/boolean_queries/outputs/experiment_1"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

config = yaml.load(open(CONFIG_PATH, "r"), Loader=yaml.SafeLoader)

model_temps = [
    f"{model} temp={temp:.1f}"
    for model in config["models"]
    for temp in config["temperatures"]
]

def process_results(df: pd.DataFrame) -> pd.DataFrame:
    is_gpt5 = df["model"].str.contains("gpt-5")
    return (
        df
        .assign(identifier=lambda df: df["question"].apply(get_question_id))
        .assign(identifier_no=lambda df: df["identifier"].str.extract(r'(\d+)').astype(int))
        .assign(model=lambda df: pd.Categorical(df["model"], categories=config["models"], ordered=True))        
        .assign(temperature=lambda df: df["temperature"].where(~is_gpt5, ""))
        .assign(**{
            "model_temp": lambda df: df["model"].astype(str).where(
                is_gpt5, 
                df["model"].astype(str) + " temp=" + df["temperature"].astype(str)
            )
        })
    )   

baseline_df = (
    pd.read_json(BOOL_DIR / "baseline_counts.jsonl", lines=True)
    .pipe(process_results)
)
logger.info(f"Baseline counts: {baseline_df.shape}")

baseline_results_df = (
    pd.read_json(BOOL_DIR / "baseline_results.jsonl", lines=True)
    .pipe(process_results)
    .drop_duplicates(subset=["identifier"], keep="first")
)
logger.info(f"Baseline results: {baseline_results_df.shape}")

llm_df = (
    pd.read_json(BOOL_DIR / "experiment1_counts.jsonl", lines=True)
    .pipe(process_results)
)
logger.info(f"Experiment 1 counts: {llm_df.shape}")

llm_results_df = (
    pd.read_json(BOOL_DIR / "experiment1_results.jsonl", lines=True)
    .pipe(process_results)
)
logger.info(f"Experiment 1 results: {llm_results_df.shape}")


In [ ]:
(
    llm_results_df
    .groupby(["identifier", "prompt"])
    .agg(counts=("question", "count"))
)

## BASELINE

In [ ]:
fig = (
    alt.Chart(baseline_df)
    .mark_bar()
    .encode(
        x="retrieved_count:Q",
        # sort by id number
        y=alt.Y("question:N", sort=alt.EncodingSortField(
            field="identifier_no",
            order="ascending"
        )),
        tooltip = [
            alt.Tooltip("identifier:N", title="Question ID"),
            alt.Tooltip("question:N", title="Question"),
            alt.Tooltip("retrieved_count:Q", title="Retrieved count"),
            alt.Tooltip("boolean_query:N", title="Boolean query"),
    ])
    .properties(
        width=600,
        height=400,
    )
)
fig

In [ ]:
fig = (
    alt.Chart(baseline_df)
    .mark_bar()
    .encode(
        x=alt.X("retrieved_count:Q", scale=alt.Scale(domain=[0, 100_000], clamp=True)),
        # sort by id number
        y=alt.Y("question:N", sort=alt.EncodingSortField(
            field="identifier_no",
            order="ascending"
        )),
        tooltip = [
            alt.Tooltip("identifier:N", title="Question ID"),
            alt.Tooltip("question:N", title="Question"),
            alt.Tooltip("retrieved_count:Q", title="Retrieved count"),
            alt.Tooltip("boolean_query:N", title="Boolean query"),
    ])
    .properties(
        width=600,
        height=400,
    )
)
fig

In [ ]:
baseline_df.head(1)

In [ ]:
fig = (
    alt.Chart(baseline_df)
    .mark_bar()
    .encode(
        x=alt.X("n_tokens:Q", bin=True),
        y=alt.Y("count()", title="Count", scale=alt.Scale(domain=[0, 15])),
    )
    .properties(
        width=250,
        height=150,
    )
)
fig

In [ ]:
fig = (
    alt.Chart(baseline_df)
    .mark_bar()
    .encode(
        x=alt.X("n_elements:Q", bin=True),
        y=alt.Y("count()", title="Count"),
    )
    .properties(
        width=250,
        height=150,
    )
)
fig

In [ ]:
baseline_df[baseline_df.n_elements == baseline_df.n_elements.max()].boolean_query.iloc[0]

In [ ]:
baseline_df.sample(1).boolean_query.iloc[0]

In [ ]:
ref = 'reference_1'
print(baseline_results_df.query("identifier == @ref").question.iloc[0])
titles = baseline_results_df.query("identifier == @ref").results_title.iloc[0]
titles[-3:]

In [ ]:
titles[-3:]

## LLM results

In [ ]:
references = ["reference_0", "reference_1", "reference_2"]
reference = references[0]
reference = "reference_5"

In [ ]:
analysis_df = (
    llm_results_df
    .query("identifier == @reference")
    .query("prompt == 'policy_atlas_v2'")
    .fillna({"retrieved_count": 0, "retrieved_total": 0})
)
len(analysis_df)

In [ ]:
pd.set_option('display.max_colwidth', 50)
(
    analysis_df
    .query("model == 'gpt-4o-mini'")
    .query("temperature == 0.5")
    [["run", "boolean_query","retrieved_total"]]
    .sort_values("run")
)

In [ ]:
config["models"]

In [ ]:
fig = (
    alt.Chart(analysis_df[["model", "temperature", "retrieved_total", "model_temp"]])
    .mark_point(size=60, opacity=0.7)
    .encode(
        x=alt.X("model_temp:N", title="Model", sort=model_temps, axis=alt.Axis(labelAngle=-45)),
        y=alt.Y("retrieved_total:Q", title="Retrieved Total", scale=alt.Scale(domain=[0, 5000], clamp=True)),
        color=alt.Color("temperature:Q", title="Temperature", scale=alt.Scale(scheme='orangered')),
    )
)
fig.save(OUTPUTS_DIR / f"retrieved_total_horizontal_{reference}.png", scale_factor=2)
fig


In [ ]:
# Base chart
base = alt.Chart(analysis_df[["model", "temperature", "retrieved_total", "model_temp"]])

# Points showing median
points = base.mark_circle(size=60, opacity=1).encode(
    x=alt.X("model_temp:N", title="Model", sort=model_temps, axis=alt.Axis(labelAngle=-45)),
    y=alt.Y("median(retrieved_total):Q", title="Retrieved Total", scale=alt.Scale(domain=[0, 1000], clamp=True)),
    color=alt.Color("temperature:Q", title="Temperature", scale=alt.Scale(scheme='orangered')),
)

# Error bars showing ± 1 standard deviation
error_bars = base.mark_errorbar(extent='iqr').encode(
    x=alt.X("model_temp:N", sort=model_temps, axis=alt.Axis(labelAngle=-45)),
    y=alt.Y("retrieved_total:Q", scale=alt.Scale(domain=[0, 5000], clamp=True)),
    color=alt.Color("temperature:Q", scale=alt.Scale(scheme='orangered')),
)

# Combine layers
fig = error_bars + points
fig.save(OUTPUTS_DIR / f"retrieved_total_median_iqr_{reference}.png", scale_factor=2)
fig


In [ ]:
exclude_models = ["gpt-5", "gpt-5-mini", "gpt-5-nano"]
mean_temp = (
    analysis_df
    .query("model not in @exclude_models")
    .groupby(["temperature"])
    .agg(
        retrieved_total_median=("retrieved_total", "median"),
        retrieved_total_q1=("retrieved_total", lambda x: x.quantile(0.25)),
        retrieved_total_q3=("retrieved_total", lambda x: x.quantile(0.75)),
    )
    .reset_index()
    .sort_values(["temperature"])
)

# Base chart
base = alt.Chart(mean_temp)

# Median points
points = base.mark_circle(size=60, opacity=1, color="red").encode(
    x=alt.X("temperature:Q", title="Temperature"),
    y=alt.Y("retrieved_total_median:Q", title="Retrieved Total"),
)

# IQR error bars
error_bars = base.mark_errorbar(color="red").encode(
    x=alt.X("temperature:Q"),
    y=alt.Y("retrieved_total_q1:Q", title=""),
    y2=alt.Y2("retrieved_total_q3:Q"),
)

fig = error_bars + points
fig = fig.properties(
    width=200,
    height=200,
    title=f"Temperature vs Retrieved Total (Median + IQR)"
)
fig.save(OUTPUTS_DIR / f"retrieved_total_temperature_median_iqr_{reference}.png", scale_factor=2)
fig

In [ ]:
def get_precision(baseline_set: list[str], retrieved_set: list[str]) -> float:
    return len(set(baseline_set) & set(retrieved_set)) / len(set(retrieved_set)) if retrieved_set else 0

def get_recall(baseline_set: list[str], retrieved_set: list[str]) -> float:
    return len(set(baseline_set) & set(retrieved_set)) / len(set(baseline_set)) if baseline_set else 0

def get_topn_recall_precision(baseline_set: list[str], retrieved_set: list[str], n: int) -> float:
    n = len(retrieved_set)
    topn_baseline_set = baseline_set[:n]
    recall = get_recall(topn_baseline_set, retrieved_set)
    precision = get_precision(topn_baseline_set, retrieved_set)
    return recall, precision

def get_metrics(baseline_set: list[str], retrieved_set: list[str]) -> tuple[float, float, float, float]:
    precision = get_precision(baseline_set, retrieved_set)
    recall = get_recall(baseline_set, retrieved_set)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
    topn_recall, topn_precision = get_topn_recall_precision(baseline_set, retrieved_set, 10)
    return precision, recall, f1, topn_recall, topn_precision


In [ ]:
baseline_result_ids = baseline_results_df.query("identifier == @reference").results_id.iloc[0]

metrics_df = analysis_df.copy()
for index, row in analysis_df.iterrows():
    test_ids = row.results_id
    if type(test_ids) == list:
        precision, recall, f1, topn_recall, topn_precision = get_metrics(baseline_result_ids, test_ids)
        metrics_df.loc[index, "precision"] = precision
        metrics_df.loc[index, "recall"] = recall
        metrics_df.loc[index, "f1"] = f1
        metrics_df.loc[index, "topn_recall"] = topn_recall
        metrics_df.loc[index, "topn_precision"] = topn_precision
    else:
        metrics_df.loc[index, "precision"] = 0
        metrics_df.loc[index, "recall"] = 0
        metrics_df.loc[index, "f1"] = 0
        metrics_df.loc[index, "topn_recall"] = 0
        metrics_df.loc[index, "topn_precision"] = 0


In [ ]:
(
    metrics_df
    .query("model == 'gpt-4o-mini'")
    .query("temperature == 0.5")
    [["run", "boolean_query","retrieved_total", "precision", "recall", "topn_recall", "topn_precision"]]
    .sort_values("run")
)

In [ ]:
# altair plot of median precision

# Base chart
base = alt.Chart(metrics_df[["model", "temperature", "f1", "model_temp"]])

# Points showing median
points = base.mark_point(size=60, opacity=1).encode(
    x=alt.X("model_temp:N", title="Model", sort=model_temps, axis=alt.Axis(labelAngle=-45)),
    y=alt.Y("f1:Q", title="F1"),
    color=alt.Color("temperature:Q", title="Temperature", scale=alt.Scale(scheme='orangered')),
)

# Combine layers
fig = points
# fig.save(OUTPUTS_DIR / f"precision_median_iqr_{reference}.png", scale_factor=2)
fig



In [ ]:
# altair plot of median precision

# Base chart
base = alt.Chart(metrics_df[["model", "temperature", "f1", "model_temp"]])

# Points showing median
points = base.mark_circle(size=60, opacity=1).encode(
    x=alt.X("model_temp:N", title="Model", sort=model_temps, axis=alt.Axis(labelAngle=-45)),
    y=alt.Y("median(f1):Q", title="F1"),
    color=alt.Color("temperature:Q", title="Temperature", scale=alt.Scale(scheme='orangered')),
)

# Error bars showing ± 1 standard deviation
error_bars = base.mark_errorbar(extent='iqr').encode(
    x=alt.X("model_temp:N", sort=model_temps, axis=alt.Axis(labelAngle=-45)),
    y=alt.Y("f1:Q", title="F1"),
    color=alt.Color("temperature:Q", scale=alt.Scale(scheme='orangered')),
)

# Combine layers
fig = error_bars + points
fig.save(OUTPUTS_DIR / f"f1_median_iqr_{reference}.png", scale_factor=2)
fig



In [ ]:
# altair plot of median precision

# Base chart
base = alt.Chart(metrics_df[["model", "temperature", "n_elements", "n_tokens", "model_temp"]])

# Points showing median
points = base.mark_circle(size=60, opacity=1).encode(
    x=alt.X("model_temp:N", title="Model", sort=model_temps, axis=alt.Axis(labelAngle=-45)),
    y=alt.Y("median(n_elements):Q", title="No of elements"),
    color=alt.Color("temperature:Q", title="Temperature", scale=alt.Scale(scheme='orangered')),
)

# Error bars showing ± 1 standard deviation
error_bars = base.mark_errorbar(extent='iqr').encode(
    x=alt.X("model_temp:N", sort=model_temps, axis=alt.Axis(labelAngle=-45)),
    y=alt.Y("n_elements:Q", title="No of elements"),
    color=alt.Color("temperature:Q", scale=alt.Scale(scheme='orangered')),
)

# Combine layers
fig = error_bars + points
fig.save(OUTPUTS_DIR / f"elements_median_iqr_{reference}.png", scale_factor=2)
fig



In [ ]:
j=2

print(
    metrics_df
    .query("model == 'gpt-5'")
    .iloc[j]["retrieved_total"]
)

(
    metrics_df
    .query("model == 'gpt-5'")
    .iloc[j]
)["boolean_query"]


# PROMPT ANALYSIS

In [ ]:
# Filter for analysis (temps 0 and 1, first 10 questions)
def filter_for_analysis(df, first_n_questions=10, prompt="policy_atlas_v2"):
    is_gpt5 = df["model"].str.contains("gpt-5")
    temp_filter = (
        ((df["temperature"] == 0.0) | (df["temperature"] == 1.0)) |
        (is_gpt5 & (df["temperature"] == ""))
    )
    return (
        df[(df["identifier_no"] < first_n_questions)]
        .query("prompt == @prompt")
        .copy()
    )

llm_filtered = filter_for_analysis(llm_results_df, prompt="policy_atlas_v1").fillna({"retrieved_total": 0})
baseline_filtered = filter_for_analysis(baseline_results_df)
print(f"Filtered to {len(llm_filtered)} LLM rows, {len(baseline_filtered)} baseline rows")

In [ ]:
# Compute F1 scores with baseline
baseline_lookup = {}
for _, row in baseline_filtered.iterrows():
    baseline_lookup[row["identifier"]] = row.get("results_id")

In [ ]:
f1_scores = []
precision_scores = []
recall_scores = []

for _, row in llm_filtered.iterrows():
    baseline_ids = baseline_lookup.get(row["identifier"])
    results_id = row.get("results_id")
    if type(results_id) is not list:
        results_id = []

    if results_id is not None and len(results_id) > 0:
        precision, recall, f1, _, _ = get_metrics(baseline_ids, set(results_id))
        f1_scores.append(f1)
        precision_scores.append(precision)
        recall_scores.append(recall)
    else:
        f1_scores.append(0)
        precision_scores.append(0)
        recall_scores.append(0)

llm_with_metrics = llm_filtered.assign(
    f1=f1_scores,
    precision=precision_scores,
    recall=recall_scores
)

In [ ]:
len(llm_with_metrics)

In [ ]:
# Aggregate by model-temperature
by_model_temp = (
    llm_with_metrics
    .groupby(["model_temp", "prompt"])
    .agg(
        n_questions=("identifier", "nunique"),
        retrieved_total_mean=("retrieved_total", "mean"),
        retrieved_total_median=("retrieved_total", "median"),
        retrieved_total_q1=("retrieved_total", lambda x: x.quantile(0.25)),
        retrieved_total_q3=("retrieved_total", lambda x: x.quantile(0.75)),
        n_elements_mean=("n_elements", "mean"),
        f1_mean=("f1", "mean"),
        precision_mean=("precision", "mean"),
        recall_mean=("recall", "mean"),
        temperature=("temperature", "first"),
    )
    .reset_index()
    .round(2)
)

In [ ]:
# Aggregate by question
by_question = (
    llm_with_metrics
    .groupby(["identifier", "identifier_no", "question"])
    .agg(
        n_model_temps=("model_temp", "nunique"),
        retrieved_total_mean=("retrieved_total", "mean"),
        retrieved_total_median=("retrieved_total", "median"),
        n_elements_mean=("n_elements", "mean"),
        f1_mean=("f1", "mean"),
        precision_mean=("precision", "mean"),
        recall_mean=("recall", "mean"),
    )
    .reset_index()
    .sort_values("identifier_no")
    .round(2)
)
by_question

In [ ]:
# Chart 1: Retrieved total by model-temp with IQR
base = alt.Chart(by_model_temp)

points = base.mark_circle(size=80, opacity=1).encode(
    x=alt.X("model_temp:N", title="Model-Temperature", axis=alt.Axis(labelAngle=-45), sort=model_temps),
    y=alt.Y("retrieved_total_median:Q", title="Retrieved Total (Median)"),
    color=alt.Color("temperature:N", title="Temperature", scale=alt.Scale(scheme='orangered')),
    tooltip=["model_temp:N", "retrieved_total_median:Q", "retrieved_total_mean:Q", "f1_mean:Q"]
)

error_bars = base.mark_errorbar().encode(
    x=alt.X("model_temp:N", axis=alt.Axis(labelAngle=-45), sort=model_temps),
    y=alt.Y("retrieved_total_q1:Q", title=""),
    y2=alt.Y2("retrieved_total_q3:Q", title=""),
    color=alt.Color("temperature:N", scale=alt.Scale(scheme='orangered')),
)

fig = (error_bars + points).properties(width=800, height=400, title="Retrieved Total by Model-Temp (Median + IQR)")
fig
# points

In [ ]:
# Chart 2: F1 scores by model-temp
fig = (
    alt.Chart(by_model_temp)
    .mark_bar(opacity=1)
    .encode(
        x=alt.X("model_temp:N", title="Model-Temperature", axis=alt.Axis(labelAngle=-45), sort=model_temps),
        y=alt.Y("f1_mean:Q", title="F1 Score"),
        color=alt.Color("temperature:Q", title="Temperature", scale=alt.Scale(scheme='orangered')),
        tooltip=["model_temp:N", "f1_mean:Q", "precision_mean:Q", "recall_mean:Q"]
    )
    .properties(width=800, height=400, title="F1 Score by Model-Temperature")
)
fig

In [ ]:
# Chart 3: F1 by question
fig = (
    alt.Chart(by_question)
    .mark_bar()
    .encode(
        y=alt.Y("question:N", title="Question", sort=alt.EncodingSortField(field="identifier_no")),
        x=alt.X("f1_mean:Q", title="F1 Score"),
        tooltip=["identifier:N", "f1_mean:Q", "retrieved_total_mean:Q"]
    )
    .properties(width=300, height=200, title="F1 Score by Question")
)
fig

In [ ]:
RESULTS_DIR = TESTING_DIR / "r_and_d/boolean_queries/outputs/experiment1"
df1 = (
    pd.read_csv(RESULTS_DIR / "policy_atlas_v2/prompt_metrics_by_question.csv")
    .assign(prompt="policy_atlas_v2")
)
df2 = (
    pd.read_csv(RESULTS_DIR / "policy_atlas_v1/prompt_metrics_by_question.csv")
    .assign(prompt="policy_atlas_v1")
)
df = pd.concat([df1, df2], ignore_index=True)

# Plot prompt1 vs prompt2 by question
fig = (
    alt.Chart(df)
    .mark_circle(opacity=1)
    .encode(
        y=alt.X("question:N", title="Question"),
        x=alt.Y("f1_median:Q", title="F1 Score"),
        color=alt.Color("prompt:N", title="Prompt", scale=alt.Scale(scheme="tableau10")),
    )
)
fig.save(RESULTS_DIR / "prompt_f1_by_question.png", scale_factor=2)

fig

In [ ]:
df1 = (
    pd.read_csv(RESULTS_DIR / "policy_atlas_v2/prompt_metrics_by_model_temp.csv")
    .assign(prompt="policy_atlas_v2")
)
df2 = (
    pd.read_csv(RESULTS_DIR / "policy_atlas_v1/prompt_metrics_by_model_temp.csv")
    .assign(prompt="policy_atlas_v1")
)
df = pd.concat([df1, df2], ignore_index=True)

# Plot F1 by model-temp
fig = (
    alt.Chart(df)
    .mark_circle(opacity=1)
    .encode(
        x=alt.X("model_temp:N", title="Model-Temperature", axis=alt.Axis(labelAngle=-45), sort=model_temps),
        y=alt.Y("f1_median:Q", title="F1 Score"),
        color=alt.Color("prompt:N", title="Prompt", scale=alt.Scale(scheme="tableau10")),
    )
)
fig.save(RESULTS_DIR / "prompt_f1_by_model_temp.png", scale_factor=2)
fig

In [ ]:
# Plot total retrieved by model-temp
fig = (
    alt.Chart(df)
    .mark_circle(opacity=1)
    .encode(
        x=alt.X("model_temp:N", title="Model-Temperature", axis=alt.Axis(labelAngle=-45), sort=model_temps),
        y=alt.Y("retrieved_total_median:Q", title="Retrieved Total (Median)"),
        color=alt.Color("prompt:N", title="Prompt", scale=alt.Scale(scheme="tableau10")),
    )
)
fig.save(RESULTS_DIR / "prompt_retrieved_total_by_model_temp.png", scale_factor=2)
fig


In [ ]:
# Plot n_elements
fig = (
    alt.Chart(df)
    .mark_circle(opacity=1)
    .encode(
        x=alt.X("model_temp:N", title="Model-Temperature", axis=alt.Axis(labelAngle=-45), sort=model_temps),
        y=alt.Y("n_elements_mean:Q", title="No of Elements (Mean)"),
        color=alt.Color("prompt:N", title="Prompt", scale=alt.Scale(scheme="tableau10")),
    )
)
fig.save(RESULTS_DIR / "prompt_n_elements_by_model_temp.png", scale_factor=2)
fig
